In [70]:
import csv
import datetime
import os

import requests
from dotenv import load_dotenv
from skyfield.api import EarthSatellite, load, wgs84

In [71]:
ts = load.timescale()

In [72]:
load_dotenv()
space_track_credentials = {}
space_track_credentials["username"] = os.getenv("SPACE_TRACK_USERNAME")
space_track_credentials["password"] = os.getenv("SPACE_TRACK_PASSWORD")

# This code will record basebands along with other info for certain satellites using the rtlsdr

In [ ]:
# variables
LAT, LON, ALT = 55.5269, 37.0888, 172
qth = wgs84.latlon(LAT, LON, elevation_m=ALT)


# Norad_cat_id, [f_center, decimation]
TARGET_SATELLITES = {
    59051: [137.9e6, 4],
    57166: [137.9e6, 4],
    25159: [137.5e6, 1],
}

In [74]:
def get_satellite_name_from_norad_id(id):
    mapping = {
        59051: "METEOR-M 2-4",
        57166: "METEOR-M 2-3",
        25159: "ORBCOMM FM04",
    }
    return mapping.get(id, "Unknown Satellite")

In [75]:
def update_tles(target_sats):
    with requests.session() as session:
        login_url = "https://www.space-track.org/ajaxauth/login"
        payload = {"identity": space_track_credentials["username"], "password": space_track_credentials["password"]}
        session.post(login_url, data=payload)
        target_sat_norads = ",".join(list(map(str, list(target_sats.keys()))))
        omm_csv = session.get(
            f"https://www.space-track.org/basicspacedata/query/class/gp/NORAD_CAT_ID/{target_sat_norads}/orderby/NORAD_CAT_ID%20asc/format/csv/emptyresult/show"
        )

    with open("OMM.csv", "w") as ommfile:
        ommfile.write(omm_csv.text)

In [76]:
def get_schedule(target_sats):
    satellites = []
    with open("OMM.csv") as csvfile:
        ommreader = csv.reader(csvfile)
        header = next(ommreader)
        for row in ommreader:
            element_dict = dict(zip(header, row, strict=True))
            if int(element_dict["NORAD_CAT_ID"]) in target_sats:
                satellites.append(EarthSatellite.from_omm(ts, element_dict))

    now = ts.now()
    t1 = now + datetime.timedelta(hours=25)
    schedule = {}
    for satellite in satellites:
        t, events = satellite.find_events(qth, now, t1, altitude_degrees=0.0)
        intervals = []
        for ti, event in zip(t, events, strict=True):
            if event == 0:
                intervals.append([ti.utc_datetime(), None])
            elif event == 2:
                try:
                    intervals[-1][1] = ti.utc_datetime()
                except IndexError:
                    print(f"Warning: Satellite {satellite.name} is currently above the horizon, omitting")
        schedule[satellite.model.satnum] = intervals

    # cleanup incomplete intervals
    for satnum, intervals in schedule.items():
        schedule[satnum] = [i for i in intervals if i[1] is not None]

    return schedule


get_schedule(TARGET_SATELLITES)

{25159: [[datetime.datetime(2026, 9, 9, 16, 57, 16, 653056, tzinfo=datetime.timezone.utc),
   datetime.datetime(2026, 9, 9, 17, 10, 19, 335111, tzinfo=datetime.timezone.utc)],
  [datetime.datetime(2026, 9, 9, 18, 33, 44, 945217, tzinfo=datetime.timezone.utc),
   datetime.datetime(2026, 9, 9, 18, 48, 24, 873576, tzinfo=datetime.timezone.utc)],
  [datetime.datetime(2026, 9, 9, 20, 12, 6, 559756, tzinfo=datetime.timezone.utc),
   datetime.datetime(2026, 9, 9, 20, 26, 34, 171184, tzinfo=datetime.timezone.utc)],
  [datetime.datetime(2026, 9, 9, 21, 53, 55, 82487, tzinfo=datetime.timezone.utc),
   datetime.datetime(2026, 9, 9, 22, 3, 34, 986322, tzinfo=datetime.timezone.utc)],
  [datetime.datetime(2026, 9, 10, 8, 17, 10, 280257, tzinfo=datetime.timezone.utc),
   datetime.datetime(2026, 9, 10, 8, 25, 51, 603239, tzinfo=datetime.timezone.utc)],
  [datetime.datetime(2026, 9, 10, 9, 53, 58, 497476, tzinfo=datetime.timezone.utc),
   datetime.datetime(2026, 9, 10, 10, 7, 58, 206701, tzinfo=datetim

In [77]:
# # Plot schedule (debugging)

# fig = plt.figure(figsize=(10, 5))
# for i, (norad_catnum, intervals) in enumerate(get_schedule(TARGET_SATELLITES).items()):
#     print(f"Satellite {i}: {get_satellite_name_from_norad_id(norad_catnum)}")
#     for interval in intervals:
#         plt.plot([interval[0], interval[1]], [i, i], linewidth=5)
# plt.xlabel('Time')
# plt.ylabel('Satellite')
# plt.title('Satellite Visibility Schedule')
# plt.grid()


In [78]:
total_daily_duration = 0
for _, intervals in get_schedule(TARGET_SATELLITES).items():
    for interval in intervals:
        duration = (interval[1] - interval[0]).total_seconds()
        total_daily_duration += duration

print(total_daily_duration / 60 / 60)  # hours

# At a sample rate of 2.4e6 with 8 bit I and Q samples, (1 byte) this is:
print(total_daily_duration * 2.4e6 * 2 / 1024 / 1024 / 1024)  # GB

6.247724465000001
100.54621729552747
